[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/09_document_structuring/09_document_structuring.ipynb)

# 09. 문서 구조화 실습 (Pydantic + OpenAI 정형 출력 + Streamlit)

> 관련 예제 프로젝트: [`example-projects/document-input-example`](../../example-projects/document-input-example) (B파트: 사용자 입력 처리)

서류 사진을 OCR로 읽으면 오타와 줄바꿈 오류가 섞인 지저분한 텍스트가 나옵니다. 이 텍스트를 그대로 검색(임베딩)에 쓰면 정확도가 떨어지므로, AI를 한 번 더 거쳐 "문서 종류, 핵심 사유, 날짜" 같은 필요한 정보만 정형화된 JSON으로 뽑아냅니다. 이 노트북에서는 그 정형화 단계에 쓰인 `pydantic`과 `openai` 라이브러리를 실습합니다.

## 이론

### 1) 왜 텍스트로 안 두고 JSON(정형 데이터)으로 바꿀까?
텍스트는 "대충 이런 내용이 있다" 정도만 알 수 있지만, JSON은 `document_type`은 무엇, `reason`은 무엇처럼 항목별로 값이 정해져 있어서 다음 단계(RAG 검색)에서 컴퓨터가 정확하게 활용하기 쉬워집니다.

### 2) `pydantic.BaseModel`
각 필드에 어떤 타입의 값이 들어가야 하는지 미리 정해두는 "양식"입니다. 이 틀에 맞춰서만 데이터를 받도록 강제할 수 있어서, 형식이 들쭉날쭉해지는 것을 막아줍니다.

### 3) OpenAI 정형 출력 (`beta.chat.completions.parse`)
"이 Pydantic 모델 형식에 딱 맞춰서 답해줘"라고 요청하면, 응답이 자동으로 해당 Pydantic 객체로 변환되어 돌아옵니다. 이 노트북은 `OPENAI_API_KEY`가 없어도 끝까지 실행되도록, 키가 없으면 간단한 규칙 기반 대체 함수로 자동 전환합니다.

### 4) `streamlit`
파이썬 코드만으로 웹 화면을 만들어주는 도구입니다. `st.file_uploader()` 같은 함수 한 줄이 화면의 버튼이 됩니다. 다만 Streamlit 앱은 `streamlit run`으로 별도 실행해야 하는 스크립트라, 노트북 안에서 직접 실행하지는 않고 코드 형태만 살펴봅니다.

## 실습 0. 환경 설정

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !pip install -q pydantic openai python-dotenv

## 실습 1. Pydantic 모델 정의하기

`structurer.py`의 `RegulationInquiry`를 그대로 재현합니다. `Field(description=...)`는 사람이 읽는 설명일 뿐 아니라, OpenAI에게 "이 필드에는 이런 내용을 채워라"라고 알려주는 힌트로도 쓰입니다.

In [ ]:
from pydantic import BaseModel, Field, ValidationError


class RegulationInquiry(BaseModel):
    document_type: str = Field(description="서류의 종류 (예: 휴가신청서, 재직증명서, 초과근무신청서 등)")
    applicant_request: str = Field(description="신청자가 요청하는 핵심 내용을 한두 문장으로 요약")
    related_dates: list[str] = Field(
        default_factory=list, description="서류에 등장하는 날짜들 (YYYY-MM-DD 형식)"
    )
    keywords: list[str] = Field(
        default_factory=list, description="규정 검색에 도움이 될 핵심 키워드 목록"
    )


print(RegulationInquiry.model_json_schema())

Pydantic은 타입이 맞지 않는 값을 넣으면 즉시 에러를 내서 잘못된 데이터가 조용히 통과하는 것을 막아줍니다.

In [ ]:
try:
    RegulationInquiry(document_type=123, applicant_request="휴가 신청", related_dates="2026-01-01")
except ValidationError as e:
    print("검증 실패:")
    print(e)

## 실습 2. OCR 원문처럼 지저분한 텍스트 준비하기

실제로는 `google-cloud-vision`이 서류 사진에서 이 텍스트를 뽑아냅니다 (Google Cloud 인증이 필요해서 이 노트북에서는 다루지 않습니다). 여기서는 OCR 결과를 흉내 낸 예시 텍스트를 준비합니다.

In [ ]:
raw_ocr_text = (
    "휴가 신청서\n"
    "성 명 : 김민준\n"
    "부 서 : 개발팀\n"
    "신청 기간 : 2026년 8월 3일 ~ 2026년 8월 7일 (5일)\n"
    "사 유 : 개인 사정으로 인한 여름 휴가 사용\n"
    "위와 같이 연차휴가를 신청합니다.\n"
    "2O26년 7월 2O일\n"  # OCR 특유의 오타: 숫자 0이 알파벳 O로 잘못 인식됨
    "신청인 : 김민준 (인)"
)
print(raw_ocr_text)

## 실습 3. OpenAI로 정형 데이터 만들기 (키가 없으면 규칙 기반 대체 함수 사용)

`OPENAI_API_KEY` 환경변수가 설정되어 있으면 실제 `gpt-4o-mini` 호출로 파싱하고, 없으면 API 비용/키 없이도 실습을 끝까지 진행할 수 있도록 아주 단순한 규칙 기반(정규식/키워드) 함수로 대체합니다.

In [ ]:
import os
import re
from dotenv import load_dotenv

load_dotenv()

SYSTEM_PROMPT = (
    "당신은 사내 서류 텍스트를 분석해 정해진 형식의 데이터로 정리하는 도우미입니다. "
    "OCR로 인식된 텍스트라 오타나 줄바꿈 오류가 있을 수 있으니, 문맥을 보고 자연스럽게 해석해 채우세요. "
    "텍스트에 없는 내용은 추측해서 지어내지 말고, 알 수 없으면 빈 값으로 두세요."
)


def structure_with_openai(raw_text: str) -> RegulationInquiry:
    from openai import OpenAI

    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    completion = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": raw_text},
        ],
        response_format=RegulationInquiry,
    )
    return completion.choices[0].message.parsed


def structure_with_rules(raw_text: str) -> RegulationInquiry:
    """API 키 없이 실습할 수 있도록 만든 아주 단순한 규칙 기반 대체 함수."""
    doc_type = "휴가신청서" if "휴가" in raw_text else "미분류 문서"

    reason_match = re.search(r"사\s*유\s*:\s*(.+)", raw_text)
    reason = reason_match.group(1).strip() if reason_match else ""

    # '2O26', '2O일'처럼 OCR이 숫자 0을 O로 잘못 읽은 경우까지 연/월/일 각 자리에서 느슨하게 잡아낸다
    date_pattern = r"([0-9oO]{4})[년.\-]\s*([0-9oO]{1,2})[월.\-]\s*([0-9oO]{1,2})"

    def fix_ocr_digits(s: str) -> str:
        return s.replace("O", "0").replace("o", "0")

    dates = []
    for y, m, d in re.findall(date_pattern, raw_text):
        y, m, d = fix_ocr_digits(y), fix_ocr_digits(m), fix_ocr_digits(d)
        dates.append(f"{y}-{int(m):02d}-{int(d):02d}")

    keywords = [kw for kw in ["연차휴가", "육아휴직", "재택근무", "경조휴가"] if kw in raw_text]
    if not keywords and doc_type == "휴가신청서":
        keywords = ["연차휴가"]

    return RegulationInquiry(
        document_type=doc_type, applicant_request=reason, related_dates=dates, keywords=keywords
    )


def structure_text(raw_text: str) -> RegulationInquiry:
    if os.getenv("OPENAI_API_KEY"):
        return structure_with_openai(raw_text)
    print("(OPENAI_API_KEY가 없어 규칙 기반 대체 함수로 실행합니다)")
    return structure_with_rules(raw_text)

## 실습 4. 결과 확인하기

In [ ]:
structured = structure_text(raw_ocr_text)
print(structured.model_dump_json(indent=2))

## 실습 5. Streamlit 앱 형태 살펴보기

`app.py`는 `st.file_uploader()`로 이미지를 받아 `structure_text()` 결과를 화면에 보여주는 구조입니다. 노트북에서 직접 실행하는 대신, 파일로 저장해두고 실제로는 터미널에서 `streamlit run app.py`로 실행합니다.

In [ ]:
app_code = '''
import streamlit as st

st.set_page_config(page_title="서류 사진 -> 정형 데이터 변환", page_icon="\U0001F4C4")
st.title("\U0001F4C4 서류 사진 -> 정형 데이터 변환")

uploaded_file = st.file_uploader("서류 이미지를 업로드하세요", type=["png", "jpg", "jpeg"])

if uploaded_file is not None:
    st.image(uploaded_file, caption="업로드한 이미지", use_container_width=True)
    if st.button("분석 시작"):
        # 실제로는 OCR(extract_text_from_image) -> structure_text() 순서로 처리합니다.
        st.write("여기에서 OCR과 정형화 결과를 보여줍니다.")
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py 저장 완료. 로컬에서는 다음 명령으로 실행합니다:")
print("  streamlit run app.py")

## 정리 & 연습 문제
- **연습 1**: `RegulationInquiry`에 `urgency`(문서 긴급도: `"높음"`/`"보통"`/`"낮음"`) 필드를 추가하고, `structure_with_rules()`에도 간단한 규칙(예: "긴급", "즉시" 같은 단어가 있으면 `"높음"`)을 추가해서 파싱 결과에 반영해보세요.
- **연습 2**: 서로 다른 서류 텍스트 3개를 리스트로 준비하고, `structure_text()`를 배치로 호출해서 `RegulationInquiry` 목록을 만드는 함수를 작성해보세요.

**해설/정답**: [09_document_structuring_solutions.ipynb](09_document_structuring_solutions.ipynb)

## 다음 단계
다음 노트북([10_rag_pipeline](../10_rag_pipeline/10_rag_pipeline.ipynb))에서는 여기서 만든 정형 정보를 활용해 규정을 검색하고 답변을 생성하는 RAG 파이프라인을 실습합니다.